In [ ]:
#  ── 1. Install deps ────────────────────────────────────────────────────────────
!pip install -q rasterio pyarrow pandas numpy scipy tqdm

# ── 2. Mount Drive ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── 3. Config ──────────────────────────────────────────────────────────────────
from pathlib import Path
import json

SRC       = Path("/content/drive/MyDrive/wildfire_viirs")
CKPT_DIR  = SRC / "checkpoints"           # one .parquet per month lives here
CKPT_LOG  = SRC / "completed_months.json" # set of finished "YYYY_MM" strings
FINAL_OUT = SRC / "viirs_spectral.parquet"

CKPT_DIR.mkdir(parents=True, exist_ok=True)
assert SRC.exists(), f"Drive folder not found: {SRC}"

print(f"Source      : {SRC}")
print(f"Checkpoints : {CKPT_DIR}")
print(f"Final out   : {FINAL_OUT}")

# ── 4. Checkpoint helpers ──────────────────────────────────────────────────────
def load_completed():
    if CKPT_LOG.exists():
        with open(CKPT_LOG) as f:
            done = set(json.load(f))
        print(f"[RESUME] {len(done)} months already done — skipping them.")
        return done
    return set()

def mark_completed(month_key, completed: set):
    completed.add(month_key)
    with open(CKPT_LOG, "w") as f:
        json.dump(sorted(completed), f)

# ── 5. Imports ─────────────────────────────────────────────────────────────────
import re, gc, shutil
from collections import defaultdict
from contextlib import contextmanager

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import rasterio
from tqdm.auto import tqdm
from scipy.spatial import cKDTree

# ── 6. Core helpers ────────────────────────────────────────────────────────────
# Band desc format: "YYYY_MM_DD_YYYYMMDD_<suffix>"  e.g. "2012_01_19_20120119_I1"
VNP09_BANDS = ["I1", "I2", "I3", "M11", "NDVI", "EVI"]
BAND_RE     = re.compile(r"^\d{4}_\d{2}_\d{2}_(\d{8})_(.+)$")

@contextmanager
def fast_local_tif(drive_path):
    local = Path("/content") / drive_path.name
    print(f"\n  -> Copying {drive_path.name} to local disk...", end="", flush=True)
    shutil.copy2(drive_path, local)
    mb = local.stat().st_size / 1024 / 1024
    print(f" {mb:.1f} MB")
    try:
        yield local
    finally:
        local.unlink(missing_ok=True)

def _coords(src):
    rows, cols = np.indices((src.height, src.width))
    lons, lats = src.transform * (cols.ravel(), rows.ravel())
    return lats.astype(np.float32), lons.astype(np.float32)

def read_bands_by_date(tif_path, wanted_suffixes):
    """Returns (lats, lons), {date_str: {suffix: arr}}"""
    day_data = defaultdict(dict)
    with rasterio.open(tif_path) as src:
        nodata = src.nodata
        coords = _coords(src)

        band_indices, band_meta = [], []
        for i in range(1, src.count + 1):
            desc = src.descriptions[i - 1]
            if not desc:
                continue
            m = BAND_RE.match(desc)
            if not m:
                continue
            date_str, suffix = m.group(1), m.group(2)
            if suffix in wanted_suffixes:
                band_indices.append(i)
                band_meta.append((date_str, suffix))

        if not band_indices:
            print(f"  [warn] no matching bands in {tif_path.name}")
            return coords, {}

        print(f"  -> Reading {len(band_indices)} bands...", end="", flush=True)
        bulk = src.read(band_indices)
        print(" Done!")

        for k, (date_str, suffix) in enumerate(tqdm(band_meta, desc="  Parsing", leave=False)):
            arr = bulk[k].ravel().astype(np.float32)
            if nodata is not None:
                arr[arr == nodata] = np.nan
            day_data[date_str][suffix] = arr

    return coords, dict(day_data)

# ── 7. LST loader for a single year (day + night) ─────────────────────────────
def load_lst_for_year(year):
    """Load LST day+night for one year into dicts keyed by date_str."""
    lst_store = {}   # date_str -> {"lst_day": arr, "lst_night": arr}
    lst_grids = {}   # date_str -> (lats, lons)

    for pattern, col in [(f"vnp21a1d_{year}.tif", "lst_day"),
                         (f"vnp21a1n_{year}.tif", "lst_night")]:
        tif = SRC / pattern
        if not tif.exists():
            print(f"  [warn] missing {pattern}")
            continue
        with fast_local_tif(tif) as local:
            coords, day_data = read_bands_by_date(local, {"LST_1KM"})
        for date_str, bands in day_data.items():
            lst_store.setdefault(date_str, {})[col] = bands["LST_1KM"]
            lst_grids.setdefault(date_str, coords)
        del day_data; gc.collect()

    print(f"  LST loaded for {len(lst_store)} dates in {year}")
    return lst_store, lst_grids

# ── 8. LST spatial join helper ─────────────────────────────────────────────────
def lst_for_date(date_str, query_lats, query_lons, lst_store, lst_grids,
                 tree_cache, max_dist=0.015):
    n = len(query_lats)
    nan_arr = lambda: np.full(n, np.nan, np.float32)

    if date_str not in lst_grids:
        return nan_arr(), nan_arr()

    lats_1km, lons_1km = lst_grids[date_str]
    key = id(lats_1km)
    if key not in tree_cache:
        valid = np.isfinite(lats_1km) & np.isfinite(lons_1km)
        pts   = np.column_stack([lats_1km[valid], lons_1km[valid]])
        tree_cache[key] = (cKDTree(pts), valid)

    tree, valid    = tree_cache[key]
    lst_cols       = lst_store.get(date_str, {})
    dists, idxs    = tree.query(np.column_stack([query_lats, query_lons]), workers=-1)
    too_far        = dists > max_dist

    out = {}
    for col in ("lst_day", "lst_night"):
        raw = lst_cols.get(col)
        if raw is None:
            out[col] = nan_arr()
            continue
        matched          = raw[valid][idxs].copy()
        matched[too_far] = np.nan
        out[col]         = matched.astype(np.float32)
    return out["lst_day"], out["lst_night"]

# ── 9. Main loop — year-outer, month-inner, checkpoint per month ───────────────
completed   = load_completed()
vnp09_tifs  = sorted(SRC.glob("vnp09ga_*.tif"))
if not vnp09_tifs:
    raise FileNotFoundError(f"No vnp09ga_*.tif in {SRC}")
print(f"\nFound {len(vnp09_tifs)} VNP09GA TIF(s)")

# Group month TIFs by year — filename: vnp09ga_YYYY_MM.tif
from collections import defaultdict as _dd
year_map = _dd(list)
for tif in vnp09_tifs:
    m = re.match(r"vnp09ga_(\d{4})_\d{2}\.tif", tif.name)
    if m:
        year_map[m.group(1)].append(tif)

total_rows  = 0
all_years   = sorted(year_map)
print(f"Years found : {all_years}")

for year in all_years:
    month_tifs = sorted(year_map[year])
    print(f"\n{'='*60}")
    print(f"  YEAR {year}  ({len(month_tifs)} month files)")
    print(f"{'='*60}")

    # ── Load this year's LST into RAM (freed at end of year loop) ──────────
    lst_store, lst_grids = load_lst_for_year(year)
    tree_cache = {}

    for tif in month_tifs:
        # month key from filename, e.g. "2012_01"
        month_key = tif.stem.replace("vnp09ga_", "")   # "2012_01"

        if month_key in completed:
            print(f"\n  [skip] {month_key} already checkpointed")
            continue

        print(f"\n--- {tif.name} ---")
        with fast_local_tif(tif) as local:
            (lats, lons), day_data = read_bands_by_date(local, set(VNP09_BANDS))

        month_rows = []

        for date_str, bands in sorted(day_data.items()):
            if len(bands) < len(VNP09_BANDS):
                print(f"  [skip] {date_str} — only {len(bands)}/{len(VNP09_BANDS)} bands")
                continue

            i1   = bands["I1"]
            mask = np.isfinite(i1)
            if not mask.any():
                print(f"  [skip] {date_str} — all NaN")
                continue

            ts = pd.Timestamp(f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}")
            lst_day, lst_night = lst_for_date(
                date_str, lats[mask], lons[mask],
                lst_store, lst_grids, tree_cache
            )

            month_rows.append(pd.DataFrame({
                "time":      ts,
                "lat":       lats[mask],
                "lon":       lons[mask],
                "i1":        i1[mask],
                "i2":        bands["I2"][mask],
                "i3":        bands["I3"][mask],
                "m11":       bands["M11"][mask],
                "ndvi":      bands["NDVI"][mask],
                "evi":       bands["EVI"][mask],
                "lst_day":   lst_day,
                "lst_night": lst_night,
            }))
            print(f"  [ok] {date_str}  {mask.sum():,} rows")

        del day_data; gc.collect()

        if not month_rows:
            print(f"  [warn] {month_key} produced no rows — marking done anyway")
            mark_completed(month_key, completed)
            continue

        df_month = pd.concat(month_rows, ignore_index=True)
        del month_rows; gc.collect()

        ckpt_path = CKPT_DIR / f"{month_key}.parquet"
        pq.write_table(
            pa.Table.from_pandas(df_month, preserve_index=False),
            ckpt_path,
            compression="zstd",
            compression_level=3,
        )
        total_rows += len(df_month)
        mark_completed(month_key, completed)
        print(f"  [saved] {month_key}.parquet  {len(df_month):,} rows  (running total {total_rows:,})")
        del df_month; gc.collect()

    # ── Free LST for this year before loading next ─────────────────────────
    del lst_store, lst_grids, tree_cache; gc.collect()
    print(f"\n  [year {year} done]  freed LST from RAM")

print(f"\n[PHASE 1 DONE]  {total_rows:,} rows across {len(completed)} months")

# ── 10. Merge all checkpoint parquets into one final file ──────────────────────
print("\nMerging checkpoint files into final parquet...")

ckpt_files = sorted(CKPT_DIR.glob("*.parquet"))
print(f"  {len(ckpt_files)} checkpoint files found")

writer      = None
merged_rows = 0

for cp in tqdm(ckpt_files, desc="Merging"):
    table = pq.read_table(cp)
    if writer is None:
        writer = pq.ParquetWriter(
            FINAL_OUT, table.schema,
            compression="zstd",
            compression_level=3,
            use_dictionary=True,
            write_statistics=True,
        )
    writer.write_table(table)
    merged_rows += len(table)

if writer:
    writer.close()

print(f"\n[DONE]  {FINAL_OUT}")
print(f"        {merged_rows:,} total rows")

# ── 11. Sanity check ───────────────────────────────────────────────────────────
df_check = pd.read_parquet(FINAL_OUT)
print(df_check.shape)
print(df_check.dtypes)
print(df_check.head(3))
